<a href="https://colab.research.google.com/github/fayefamudulan/industrial-dashboard/blob/main/COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pyngrok plotly seaborn scikit-learn openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 75.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

data = {
    'Date': pd.date_range(start='2026-01-01', periods=100),
    'Temperature': np.random.uniform(60, 90, 100),
    'Pressure': np.random.uniform(80, 120, 100),
    'Speed': np.random.uniform(1200, 1800, 100),
    'Production_Output': np.random.randint(200, 500, 100)
}
df = pd.DataFrame(data)
df.to_csv('industrial_data.csv', index=False)
print("CSV File Created!")

CSV File Created!


In [4]:
from google.colab import files
uploaded = files.upload()

Saving industrial_data.csv to industrial_data (1).csv


In [5]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# ======================================================
# PAGE CONFIG
# ======================================================

st.set_page_config(
    page_title="Industrial Engineering Dashboard",
    page_icon="🏭",
    layout="wide"
)

# ======================================================
# CUSTOM CSS
# ======================================================

st.markdown("""
<style>

.main {
    background-color: #f5f7fa;
}

.metric-card {
    background: white;
    padding: 20px;
    border-radius: 15px;
    box-shadow: 0px 2px 10px rgba(0,0,0,0.1);
}

h1,h2,h3 {
    color: #1f2937;
}

</style>
""", unsafe_allow_html=True)

# ======================================================
# HEADER
# ======================================================

st.title("🏭 Industrial Engineering Production Dashboard")
st.markdown("### Real-Time Forecasting • SPC Monitoring • Production Analytics")

# ======================================================
# SIDEBAR
# ======================================================

st.sidebar.header("⚙ Dashboard Controls")

uploaded_file = st.sidebar.file_uploader(
    "Upload CSV Dataset",
    type=["csv"]
)

if uploaded_file is None:
    st.warning("Please upload a CSV dataset.")
    st.stop()

# ======================================================
# LOAD DATA
# ======================================================

df = pd.read_csv(uploaded_file)

st.subheader("📄 Dataset Preview")

st.dataframe(df.head())

# ======================================================
# NUMERIC COLUMNS
# ======================================================

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

if len(numeric_cols) < 2:
    st.error("Dataset needs at least 2 numeric columns.")
    st.stop()

target = st.sidebar.selectbox(
    "🎯 Select Target Variable",
    numeric_cols
)

features = [col for col in numeric_cols if col != target]

# ======================================================
# FILTERS
# ======================================================

st.sidebar.subheader("📊 Data Filters")

rows_to_show = st.sidebar.slider(
    "Rows to Display",
    5,
    min(100, len(df)),
    10
)

# ======================================================
# DATA SUMMARY
# ======================================================

st.subheader("📈 Data Summary")

col1, col2, col3, col4 = st.columns(4)

col1.metric("Rows", len(df))
col2.metric("Columns", len(df.columns))
col3.metric("Features", len(features))
col4.metric("Target", target)

# ======================================================
# CORRELATION HEATMAP
# ======================================================

st.subheader("🔥 Correlation Heatmap")

corr = df[numeric_cols].corr()

fig_corr = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale="Blues"
)

st.plotly_chart(fig_corr, use_container_width=True)

# ======================================================
# MODEL TRAINING
# ======================================================

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = LinearRegression()

model.fit(X_train, y_train)

predictions = model.predict(X_test)

# ======================================================
# METRICS
# ======================================================

r2 = r2_score(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)

st.subheader("🧠 Model Performance")

m1, m2, m3 = st.columns(3)

m1.metric("R² Score", f"{r2:.4f}")
m2.metric("RMSE", f"{rmse:.4f}")
m3.metric("MAE", f"{mae:.4f}")

# ======================================================
# ACTUAL VS PREDICTED
# ======================================================

st.subheader("📉 Actual vs Predicted")

actual_pred_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": predictions
})

fig_actual = px.scatter(
    actual_pred_df,
    x="Actual",
    y="Predicted",
    trendline="ols",
    color="Predicted",
    title="Actual vs Predicted Production"
)

st.plotly_chart(fig_actual, use_container_width=True)

# ======================================================
# RESIDUAL ANALYSIS
# ======================================================

residuals = y_test.values - predictions

mean_res = np.mean(residuals)
std_res = np.std(residuals)

ucl = mean_res + (3 * std_res)
lcl = mean_res - (3 * std_res)

residual_df = pd.DataFrame({
    "Index": range(len(residuals)),
    "Residuals": residuals
})

residual_df["Anomaly"] = (
    (residual_df["Residuals"] > ucl) |
    (residual_df["Residuals"] < lcl)
)

# ======================================================
# SPC CHART
# ======================================================

st.subheader("🚨 SPC Residual Control Chart")

fig_spc = go.Figure()

fig_spc.add_trace(
    go.Scatter(
        x=residual_df["Index"],
        y=residual_df["Residuals"],
        mode='lines+markers',
        name='Residuals'
    )
)

fig_spc.add_hline(
    y=ucl,
    line_dash="dash",
    line_color="red",
    annotation_text="UCL"
)

fig_spc.add_hline(
    y=lcl,
    line_dash="dash",
    line_color="red",
    annotation_text="LCL"
)

fig_spc.add_hline(
    y=mean_res,
    line_dash="dash",
    line_color="green",
    annotation_text="Mean"
)

st.plotly_chart(fig_spc, use_container_width=True)

# ======================================================
# ANOMALY DETECTION
# ======================================================

st.subheader("⚠ Detected Anomalies")

anomalies = residual_df[residual_df["Anomaly"] == True]

if len(anomalies) > 0:
    st.dataframe(anomalies)
else:
    st.success("No anomalies detected.")

# ======================================================
# FEATURE IMPORTANCE
# ======================================================

st.subheader("📌 Feature Coefficients")

coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_
})

coef_df = coef_df.sort_values(
    by="Coefficient",
    ascending=False
)

fig_coef = px.bar(
    coef_df,
    x="Coefficient",
    y="Feature",
    orientation='h',
    color="Coefficient",
    title="Feature Impact on Production"
)

st.plotly_chart(fig_coef, use_container_width=True)

# ======================================================
# DISTRIBUTION PLOT
# ======================================================

st.subheader("📊 Target Distribution")

fig_dist = px.histogram(
    df,
    x=target,
    nbins=30,
    color_discrete_sequence=["#2563eb"]
)

st.plotly_chart(fig_dist, use_container_width=True)

# ======================================================
# RAW DATA
# ======================================================

st.subheader("🗂 Raw Dataset")

st.dataframe(df.head(rows_to_show))

# ======================================================
# DOWNLOAD DATA
# ======================================================

csv = df.to_csv(index=False).encode('utf-8')

st.download_button(
    label="⬇ Download Dataset",
    data=csv,
    file_name='filtered_dataset.csv',
    mime='text/csv'
)

# ======================================================
# FOOTER
# ======================================================

st.markdown("---")

st.markdown("""
### ✅ Dashboard Features
- Linear Regression Forecasting
- SPC Control Charts
- Residual Monitoring
- Anomaly Detection
- Correlation Analysis
- Interactive Visualizations
- Feature Importance Analytics
- CSV Export
""")

Writing app.py


In [6]:
!streamlit run app.py &>/content/logs.txt &

In [9]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 22 packages in 4s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴npm notice
npm notice New major version of npm available! 10.8.2 -> 11.14.0
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.14.0
npm notice To update run: npm install -g npm@11.14.0
npm notice
⠴

In [10]:
!streamlit run app.py &>/content/logs.txt &

In [14]:
!pip install gradio

In [17]:
!streamlit run app.py &>/content/logs.txt &

In [21]:
import plotly.express as px

In [27]:
from google.colab import files